In [ ]:
%%bash
if [ -d "./cifar10_data/cifar-10-batches-py" ] && [ -f "./cifar10_data/cifar-10-batches-py/batches.meta" ]; then
    echo "Dataset already exists, skipping download."
else
    echo "Dataset not found, downloading..."
    kaggle datasets download -d pankrzysiu/cifar10-python --force
    unzip -q cifar10-python.zip -d ./cifar10_data
fi

In [ ]:
import pickle
import numpy as np
os = __import__('os')


def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict


data_dir = './cifar10_data/cifar-10-batches-py/'

x_train_list = []
y_train_list = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    batch = unpickle(batch_path)
    x_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])

x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

test_batch = unpickle(os.path.join(data_dir, 'test_batch'))
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

meta_batch = unpickle(os.path.join(data_dir, 'batches.meta'))
classes = [b.decode('utf-8') for b in meta_batch[b'label_names']]

print(f"Training features shape: {x_train.shape}")
print(f"Training labels shape:   {y_train.shape}")
print(f"Testing features shape:  {x_test.shape}")
print(f"Dataset classes:         {classes}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

x_train_images = x_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
x_test_images = x_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

sample_image = x_train_images[0]
print("Image Shape:", sample_image.shape)
print("Number of Channels:", sample_image.shape[2])

### Task 1: Image Representation
- **Pixel Values**: Integers from 0 to 255 representing brightness.
- **RGB Channel Representation**: Three matrices (Red, Green, Blue channels) representing color intensities.

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(sample_image)
plt.title(f"Class: {classes[y_train[0]]}")
plt.axis('off')
plt.show()

In [ ]:
gray_image = cv2.cvtColor(sample_image, cv2.COLOR_RGB2GRAY)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(sample_image)
plt.title("Original RGB")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gray_image, cmap='gray')
plt.title("Grayscale")
plt.axis('off')
plt.show()

### Task 3: RGB vs Grayscale Difference
- **RGB**: Uses three color channels (Red, Green, Blue) to represent colors.
- **Grayscale**: Uses a single channel to represent intensity/luminance only.

In [ ]:
resized_image = cv2.resize(sample_image, (128, 128), interpolation=cv2.INTER_LINEAR)

center = (16, 16)
matrix = cv2.getRotationMatrix2D(center, 45, 1.0)
rotated_image = cv2.warpAffine(sample_image, matrix, (32, 32))

flipped_image = cv2.flip(sample_image, 1)

plt.figure(figsize=(10, 3))
plt.subplot(1, 3, 1)
plt.imshow(resized_image)
plt.title("Resized (128x128)")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(rotated_image)
plt.title("Rotated (45°)")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(flipped_image)
plt.title("Horizontal Flip")
plt.axis('off')
plt.show()

In [ ]:
gaussian_blur = cv2.GaussianBlur(sample_image, (3, 3), 0)
median_blur = cv2.medianBlur(sample_image, 3)

plt.figure(figsize=(10, 3))
plt.subplot(1, 3, 1)
plt.imshow(sample_image)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(gaussian_blur)
plt.title("Gaussian Blur")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(median_blur)
plt.title("Median Blur")
plt.axis('off')
plt.show()

### Task 5: How Filtering Reduces Noise
- Filtering convolved the image with a spatial filter kernel to smooth out high-frequency variations (noise) by averaging neighboring pixel values.

In [ ]:
edges_original = cv2.Canny(gray_image, 100, 200)
edges_blurred = cv2.Canny(cv2.cvtColor(gaussian_blur, cv2.COLOR_RGB2GRAY), 50, 150)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(edges_original, cmap='gray')
plt.title("Edges (Original)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(edges_blurred, cmap='gray')
plt.title("Edges (Blurred)")
plt.axis('off')
plt.show()

### Task 6: Edge Detection Concept
- Edge detection identifies boundaries of objects where there is a sharp/significant change in pixel intensity.

In [ ]:
pipeline_image = x_train_images[1]
pipeline_gray = cv2.cvtColor(pipeline_image, cv2.COLOR_RGB2GRAY)
pipeline_blur = cv2.GaussianBlur(pipeline_gray, (3, 3), 0)
pipeline_edges = cv2.Canny(pipeline_blur, 50, 150)

plt.figure(figsize=(3, 3))
plt.imshow(pipeline_edges, cmap='gray')
plt.title("Pipeline Output")
plt.axis('off')
plt.show()

In [ ]:
plt.figure(figsize=(12, 3))
plt.subplot(1, 5, 1)
plt.imshow(sample_image)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 5, 2)
plt.imshow(gray_image, cmap='gray')
plt.title("Grayscale")
plt.axis('off')

plt.subplot(1, 5, 3)
plt.imshow(resized_image)
plt.title("Resized")
plt.axis('off')

plt.subplot(1, 5, 4)
plt.imshow(gaussian_blur)
plt.title("Blurred")
plt.axis('off')

plt.subplot(1, 5, 5)
plt.imshow(edges_blurred, cmap='gray')
plt.title("Edges")
plt.axis('off')
plt.show()